---
title: "Machine Learning: Graphical Models and Sequential Prediction"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    code-fold: true
jupyter: python
---


<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/11-graphical-sequential-models.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Graphical Models and Sequential Prediction**

Many machine-learning problems do not ask for one independent label. A medical model reasons about interacting symptoms and diseases; a speech recognizer must produce an ordered word sequence; a named-entity recognizer assigns labels whose validity depends on neighboring labels; and a tracking system estimates a hidden state that evolves through time. In each case, the unknown answer has **internal structure**.

A **probabilistic graphical model (PGM)** represents a joint probability distribution with a graph. Variables appear as nodes, while edges and factors state which variables interact directly. The graph is not merely a visualization of correlation. Under the model's assumptions, graph separation corresponds to conditional independence, and conditional independence allows a large joint distribution to be written as a product of smaller local functions.

This chapter connects four questions that should be kept distinct:

1. **Representation:** which dependencies and independences does the model assume?
2. **Inference:** after observing evidence, how are marginals, likelihoods, or the most likely structured output computed?
3. **Learning:** how are local probabilities, potentials, or feature weights estimated from data?
4. **Decision:** should prediction minimize token errors, sequence errors, or a task-specific structured loss?

The same graph may support several inference tasks. For latent variables $Z$, observations $X$, and a structured target $Y$, common requests include

$$
p(Z\mid X), \qquad p(X), \qquad
\hat{Y}_{\mathrm{MAP}}=\arg\max_Y p(Y\mid X),
\qquad
\mathbb E[g(Z)\mid X].
$$

These are respectively posterior inference, evidence evaluation, maximum a posteriori decoding, and a posterior expectation. They require different algorithms even when they use the same model.

<div class="diagram-scroll">

![A map from a joint probability distribution to Bayesian networks, Markov random fields, factor graphs, hidden Markov models, and conditional random fields.](assets/graphical-model-overview.svg){fig-alt="Graphical model families expressed as alternative structural descriptions of joint or conditional distributions."}

</div>

The chapter begins with general graphical-model language and then specializes it to sequences. A Bayesian network gives a directed generative factorization; a Markov random field gives an undirected compatibility-based factorization; and a factor graph makes individual factors explicit. Hidden Markov models (HMMs) add a repeated latent-state structure, while conditional random fields (CRFs) directly model a structured label distribution conditioned on observed input. Structured perceptrons and structured support vector machines keep the structured decoder but replace normalized probability with discriminative objectives.

| Model family | Distribution modeled | Latent variables | Normalized probability | Typical use |
|---|---|---:|---:|---|
| Bayesian network | Joint $p(\mathbf{x})$ | Optional | Yes | Causal assumptions, diagnosis, missing data |
| Markov random field | Joint $p(\mathbf{x})$ | Optional | Yes | Spatial compatibility, collective labeling |
| HMM | Joint $p(\mathbf{x},\mathbf{z})$ | Central | Yes | Segmentation, state tracking, sequence likelihood |
| Linear-chain CRF | Conditional $p(\mathbf{y}\mid\mathbf{x})$ | Usually no | Yes | Sequence labeling with overlapping features |
| Structured perceptron / SVM | Score $s(\mathbf{x},\mathbf{y})$ | Usually no | No | Large-output prediction with a task-aware decoder |

The key intellectual move is to stop treating a structured model as one enormous table. Structure says which local computations can be reused. On a chain, this produces dynamic programming; on a tree, exact message passing; on a graph with many loops, exact inference may become exponential and approximation becomes necessary.


### **Why Represent Probability with a Graph?**

Suppose $X_1,\ldots,X_d$ are binary. An unrestricted joint distribution needs $2^d-1$ free probabilities after normalization. With only $d=30$, the table already has more than one billion entries. A graph controls this growth by asserting that each variable interacts directly with only a small neighborhood. The resulting factorization can reduce both the number of parameters and the cost of inference.

A graphical model therefore performs two jobs at once:

- it is a **statistical hypothesis** about which dependencies matter;
- it is a **computational blueprint** showing how a global probability or score decomposes into reusable local terms.

The benefit is not free. If a missing edge incorrectly removes a real dependency, the model is biased. If too many edges are added, estimation needs more data and inference becomes harder. Graph design is consequently a form of inductive bias, much like choosing linearity, smoothness, or a neural-network architecture.

#### **Nodes, Edges, Factors, and Conditional Independence**

A **node** represents a random variable or a group of variables. An **edge** indicates a direct dependency according to the semantics of the graph family. A **factor** is a non-negative local function over a subset of variables. Factors need not be probabilities by themselves; their product becomes a normalized probability after division by a partition function.

Conditional independence is written

$$
X \perp Y \mid Z,
$$

meaning that once $Z$ is known, learning $Y$ does not change the conditional distribution of $X$:

$$
p(x\mid y,z)=p(x\mid z)
$$

whenever the conditioning event has positive probability. This is stronger than zero correlation. Independence concerns the entire conditional distribution, whereas correlation measures only a particular form of association.

Directed graphs contain three local motifs that explain much of their behavior:

<div class="diagram-scroll">

![Chain, fork, and collider motifs showing when graph paths are active or blocked by conditioning.](assets/conditional-independence-motifs.svg){fig-alt="The chain, fork, and collider motifs used to reason about d-separation and explaining away."}

</div>

- In a **chain**, $X\rightarrow Z\rightarrow Y$, conditioning on $Z$ blocks the path, so $X\perp Y\mid Z$.
- In a **fork**, $X\leftarrow Z\rightarrow Y$, the common cause $Z$ explains the association. Conditioning on $Z$ again blocks the path.
- In a **collider**, $X\rightarrow Z\leftarrow Y$, the path is blocked before $Z$ is observed. Conditioning on $Z$, or on one of its descendants, opens the path and can make $X$ and $Y$ dependent. This is the source of **explaining away**.

For example, a security alarm may be triggered by burglary or earthquake. Before hearing the alarm, those causes may be independent. After hearing it, evidence for an earthquake makes burglary less necessary as an explanation, so the causes become dependent conditional on the alarm.

<details>
<summary><strong>Python: observe explaining away by exact enumeration</strong></summary>

```python
from itertools import product

# B = burglary, E = earthquake, A = alarm; all variables are binary.
p_b = {0: 0.99, 1: 0.01}
p_e = {0: 0.98, 1: 0.02}
p_alarm_1 = {
    (0, 0): 0.001,
    (1, 0): 0.95,
    (0, 1): 0.80,
    (1, 1): 0.99,
}

joint = {}
for b, e, a in product([0, 1], repeat=3):
    p_a = p_alarm_1[(b, e)] if a else 1 - p_alarm_1[(b, e)]
    joint[(b, e, a)] = p_b[b] * p_e[e] * p_a

def conditional_probability(query, evidence):
    """Sum exact joint probabilities matching query and evidence."""
    numerator = 0.0
    denominator = 0.0
    names = {"B": 0, "E": 1, "A": 2}
    for assignment, probability in joint.items():
        if all(assignment[names[k]] == v for k, v in evidence.items()):
            denominator += probability
            if all(assignment[names[k]] == v for k, v in query.items()):
                numerator += probability
    return numerator / denominator

print("P(B=1)             =", round(conditional_probability({"B": 1}, {}), 4))
print("P(B=1 | A=1)       =", round(conditional_probability({"B": 1}, {"A": 1}), 4))
print("P(B=1 | A=1,E=1)   =", round(conditional_probability({"B": 1}, {"A": 1, "E": 1}), 4))
```

</details>

The alarm raises the probability of burglary. Once an earthquake is also known, the burglary probability falls because the earthquake explains part of the alarm evidence. The arithmetic illustrates why conditioning cannot be understood as simply deleting uncertainty: depending on graph structure, it may block one path and open another.

Graphical separation is a property of the assumed graph, not proof that the real world obeys that independence. Domain knowledge, data, and sensitivity analysis must still justify the model. The graph makes assumptions inspectable; it does not make them automatically correct.


### **Bayesian Networks**

A **Bayesian network (BN)** represents a joint distribution with a directed acyclic graph (DAG). The direction of an arrow determines the mathematical factorization: every node is modeled conditionally on its parents. Direction can also encode a causal hypothesis, but only when the graph and data-collection assumptions justify a causal interpretation. A directed edge by itself means probabilistic dependence in a factorization, not automatically causation.

#### **Directed Acyclic Graphs**

A graph is directed when every edge has an arrow, and acyclic when no directed path returns to its starting node. A DAG admits at least one **topological order** in which every parent appears before its children. That order gives a generative procedure:

1. sample each root from its marginal distribution;
2. visit remaining nodes in topological order;
3. sample each node from its conditional distribution given its sampled parents.

For variables $X_1,\ldots,X_d$ with parent sets $\mathrm{Pa}(X_i)$, the network factorization is

$$
p(x_1,\ldots,x_d)
=\prod_{i=1}^{d}p\!\left(x_i\mid x_{\mathrm{Pa}(i)}\right).
$$

The ordinary chain rule can factor any joint distribution, but a BN becomes useful when the parent sets are small. If $X_i$ is binary and has $k$ binary parents, its full conditional probability table needs $2^k$ probabilities instead of conditioning on all preceding variables.

Consider a simple medical network:

$$
S\rightarrow D, \qquad D\rightarrow T, \qquad D\rightarrow C,
$$

where $S$ is smoking, $D$ is disease, $T$ is a test result, and $C$ is a symptom. Its joint distribution is

$$
p(s,d,t,c)=p(s)p(d\mid s)p(t\mid d)p(c\mid d).
$$

The factorization says that after disease status is known, the test and symptom are conditionally independent of smoking and of each other. This local assumption lets diagnosis combine evidence without defining a four-dimensional table directly.

<details>
<summary><strong>Python: sample and query a small Bayesian network</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(7)
n = 100_000

# Sample in topological order: Smoking -> Disease -> {Test, Symptom}.
smoking = rng.binomial(1, 0.25, size=n)
p_disease = np.where(smoking == 1, 0.12, 0.02)
disease = rng.binomial(1, p_disease)
test = rng.binomial(1, np.where(disease == 1, 0.90, 0.08))
symptom = rng.binomial(1, np.where(disease == 1, 0.75, 0.10))

def empirical_probability(mask, condition=None):
    if condition is None:
        return mask.mean()
    return mask[condition].mean()

print("P(D=1)             =", round(empirical_probability(disease == 1), 4))
print("P(D=1 | T=1)       =", round(empirical_probability(disease == 1, test == 1), 4))
print("P(D=1 | T=1,C=1)   =", round(
    empirical_probability(disease == 1, (test == 1) & (symptom == 1)), 4
))
```

</details>

The code separates the model from the query. The sampling lines define local mechanisms; the masks approximate posterior questions. Exact inference would sum the same local factors rather than rely on Monte Carlo samples.

#### **Factorization and D-Separation**

**D-separation** is the graphical rule for deciding whether sets of nodes $A$ and $B$ are conditionally independent given evidence $C$ in a DAG. A path is active if every non-collider on the path is unobserved and every collider has itself or a descendant in the conditioning set. If all paths between $A$ and $B$ are blocked, then

$$
A\perp B\mid C
$$

for every distribution that factorizes according to the graph.

Two qualifications matter:

- D-separation implies independence under the model; it does not claim that all independences in a particular parameter setting appear in the graph.
- Different DAGs can imply exactly the same conditional independences. Such graphs are **Markov equivalent**, so observational data alone may not identify arrow direction.

A practical reasoning sequence is: list every undirected path between the queried nodes, classify each intermediate node as a chain/fork or collider along that path, then apply the blocking rules using the evidence set. This is safer than reasoning from arrows informally.

#### **Learning Parameters and Structure**

When a discrete DAG is fixed and every variable is observed, maximum-likelihood parameter learning reduces to local counting. For node $X_i$, parent configuration $u$, and state $k$,

$$
\widehat{p}(X_i=k\mid \mathrm{Pa}(X_i)=u)
=\frac{N_{i,k,u}}{N_{i,u}}.
$$

Sparse configurations create zero probabilities, so a Dirichlet prior or additive smoothing is often used:

$$
\widehat{p}(X_i=k\mid u)
=\frac{N_{i,k,u}+\alpha_k}
{N_{i,u}+\sum_j\alpha_j}.
$$

The prior has a concrete interpretation: $\alpha_k$ acts like a pseudo-count for state $k$. It prevents one unseen local event from forcing the probability of an entire future assignment to zero.

<details>
<summary><strong>Python: estimate a conditional probability table with smoothing</strong></summary>

```python
import numpy as np

# Rows are [smoking, disease], including a deliberately small data set.
observations = np.array([
    [0, 0], [0, 0], [0, 0], [0, 1],
    [1, 0], [1, 1], [1, 1],
])

def binary_cpt(data, parent_column, child_column, alpha=1.0):
    table = np.zeros((2, 2), dtype=float)
    for parent_value in (0, 1):
        child_values = data[data[:, parent_column] == parent_value, child_column]
        counts = np.bincount(child_values, minlength=2)
        # A symmetric Beta(alpha, alpha) prior gives one pseudo-count per state.
        table[parent_value] = (counts + alpha) / (counts.sum() + 2 * alpha)
    return table

cpt = binary_cpt(observations, parent_column=0, child_column=1, alpha=1.0)
print("Rows: smoking=0/1; columns: disease=0/1")
print(np.round(cpt, 3))
```

</details>

If variables are missing or latent, local counts are unavailable. Expectation-maximization can alternate between posterior inference over missing variables and expected-count parameter updates. Fully Bayesian learning instead places distributions over local parameters and integrates or samples them.

**Structure learning** searches over DAGs. A score-based method chooses a graph $G$ using a penalized fit such as BIC:

$$
\mathrm{BIC}(G)
=\log p(\mathcal D\mid\widehat\theta_G,G)
-\frac{k_G}{2}\log n,
$$

where $k_G$ is the number of free parameters. The likelihood rewards fit; the penalty discourages unnecessary edges. Constraint-based methods instead test conditional independences and construct a compatible graph. Hybrid methods combine both ideas.

Structure learning is difficult because the number of DAGs grows super-exponentially and finite-sample independence tests are imperfect. Greedy edge addition, deletion, and reversal are common, but they can stop at local optima. Domain constraints such as temporal order or forbidden edges often improve both plausibility and search efficiency.

<details>
<summary><strong>Python: compare two candidate DAG structures with BIC</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(3)
n = 600
x = rng.binomial(1, 0.5, n)
y = rng.binomial(1, np.where(x == 1, 0.85, 0.15))
data = np.column_stack([x, y])

def bernoulli_log_likelihood(values):
    p = np.clip(values.mean(), 1e-9, 1 - 1e-9)
    return np.sum(values * np.log(p) + (1 - values) * np.log(1 - p))

def bic_independent(binary_data):
    # p(X)p(Y): two Bernoulli parameters.
    log_likelihood = sum(bernoulli_log_likelihood(binary_data[:, j]) for j in range(2))
    return log_likelihood - 0.5 * 2 * np.log(len(binary_data))

def bic_x_to_y(binary_data):
    # p(X)p(Y|X): one parameter for X and one for each value of X.
    log_likelihood = bernoulli_log_likelihood(binary_data[:, 0])
    for x_value in (0, 1):
        log_likelihood += bernoulli_log_likelihood(
            binary_data[binary_data[:, 0] == x_value, 1]
        )
    return log_likelihood - 0.5 * 3 * np.log(len(binary_data))

print("BIC, no edge:", round(bic_independent(data), 2))
print("BIC, X -> Y :", round(bic_x_to_y(data), 2))
```

</details>

The edge adds a parameter, so it must improve log-likelihood enough to pay the complexity penalty. Notice that observational BIC cannot distinguish $X\rightarrow Y$ from $X\leftarrow Y$ for two variables: both encode the same independence structure. Direction requires additional variables, temporal or experimental knowledge, or stronger assumptions.


### **Markov Random Fields and Factor Graphs**

A **Markov random field (MRF)**, also called a Markov network, uses an undirected graph. It is natural when dependence is symmetric or when specifying a causal/generative order is awkward. Neighboring image pixels, interacting labels on a map, and compatible assignments in a constraint system are examples where “agreement” is easier to express than one variable generating another.

In an MRF, a node is conditionally independent of all non-neighbors given its neighbors. For positive distributions, the Hammersley-Clifford theorem connects this local Markov property to factorization over graph cliques.

#### **Undirected Factorization**

Let $\mathcal C$ be a collection of cliques and let $\psi_C(x_C)\geq 0$ be a potential for each clique. Then

$$
p(\mathbf{x})
=\frac{1}{Z}\prod_{C\in\mathcal C}\psi_C(x_C),
\qquad
Z=\sum_{\mathbf{x}}\prod_{C\in\mathcal C}\psi_C(x_C).
$$

The potential $\psi_C$ measures compatibility, not a locally normalized conditional probability. The **partition function** $Z$ sums the unnormalized score over all assignments and makes the global distribution sum to one. If $m$ binary variables are coupled, direct computation of $Z$ may require $2^m$ terms; this normalization is a major source of computational difficulty.

A **factor graph** is a bipartite representation with variable nodes on one side and factor nodes on the other. It makes the factorization more explicit than an ordinary undirected graph: a factor node connects exactly to the variables in its scope. Factor graphs are especially useful for deriving message-passing algorithms because messages alternate between variables and factors.

<div class="diagram-scroll">

![An undirected Markov random field, its equivalent factor graph, and the corresponding energy landscape.](assets/mrf-factor-graph-energy.svg){fig-alt="A pairwise Markov random field rewritten as a bipartite factor graph and as an energy-based probability model."}

</div>

#### **Energy-Based Representations**

Writing each potential as an exponential produces an **energy-based model**:

$$
p(\mathbf{x})
=\frac{\exp[-E(\mathbf{x})]}{Z},
\qquad
Z=\sum_{\mathbf{x}}\exp[-E(\mathbf{x})].
$$

Low-energy assignments receive high probability. For an Ising-style binary model with spins $x_i\in\{-1,+1\}$,

$$
E(\mathbf{x})
=-\sum_i h_i x_i-\sum_{(i,j)\in E}J_{ij}x_ix_j.
$$

The field $h_i$ expresses a local preference. A positive coupling $J_{ij}$ rewards equal neighboring spins because $x_ix_j=+1$; a negative coupling rewards disagreement. The partition function couples every local choice globally.

<details>
<summary><strong>Python: enumerate a small Ising model exactly</strong></summary>

```python
from itertools import product
import numpy as np

# Four variables arranged in a chain. Positive J encourages adjacent agreement.
states = np.array(list(product([-1, 1], repeat=4)))
h = np.array([0.2, -0.1, 0.0, 0.3])
J = np.array([0.8, 0.8, 0.8])

def energy(x):
    local = -np.dot(h, x)
    pairwise = -np.sum(J * x[:-1] * x[1:])
    return local + pairwise

energies = np.array([energy(x) for x in states])
unnormalized = np.exp(-energies)
probabilities = unnormalized / unnormalized.sum()

marginal_probability_plus = probabilities @ (states == 1)
best_state = states[np.argmax(probabilities)]

print("Partition function Z:", round(unnormalized.sum(), 3))
print("P(X_i=+1):", np.round(marginal_probability_plus, 3))
print("MAP state:", best_state.tolist())
```

</details>

Exact enumeration is useful as a correctness oracle for tiny models. It quickly becomes impossible as the state space grows, which motivates variable elimination, message passing, sampling, and variational approximation.

Sampling from an MRF can exploit its local conditional distributions. In the binary Ising model, the conditional probability for one spin depends only on its neighbors:

$$
p(x_i=+1\mid\mathbf{x}_{-i})
=\sigma\!\left(2h_i+2\sum_{j\in N(i)}J_{ij}x_j\right),
$$

where $\sigma(a)=1/(1+e^{-a})$. Gibbs sampling repeatedly draws each variable from this local conditional, avoiding direct calculation of $Z$.

<details>
<summary><strong>Python: Gibbs sampling using only the Markov blanket</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(12)
h = np.array([0.2, -0.1, 0.0, 0.3])
J = np.array([0.8, 0.8, 0.8])
state = rng.choice([-1, 1], size=4)
samples = []

for sweep in range(12_000):
    for i in range(4):
        neighbor_field = h[i]
        if i > 0:
            neighbor_field += J[i - 1] * state[i - 1]
        if i < 3:
            neighbor_field += J[i] * state[i + 1]

        probability_plus = 1 / (1 + np.exp(-2 * neighbor_field))
        state[i] = 1 if rng.random() < probability_plus else -1

    # Discard burn-in and thin the correlated chain.
    if sweep >= 2_000 and sweep % 5 == 0:
        samples.append(state.copy())

samples = np.asarray(samples)
print("Gibbs estimate of P(X_i=+1):", np.round((samples == 1).mean(axis=0), 3))
```

</details>

The algorithm is local, but its samples are correlated. Strong couplings can make the chain remain in one mode for a long time. More samples do not fix poor mixing automatically; diagnostics and alternative samplers may be needed.

| Property | Bayesian network | Markov random field | Factor graph |
|---|---|---|---|
| Edge semantics | Directed parent-child dependence | Undirected compatibility | Variable-factor membership |
| Local object | Conditional probability | Clique potential | Explicit arbitrary factor |
| Normalization | Local conditionals imply global normalization | Requires global $Z$ | Depends on represented model |
| Natural question | How could variables be generated? | Which assignments are mutually compatible? | How can the product be computed efficiently? |
| Typical challenge | DAG structure and latent inference | Partition function and loops | Message size and graph topology |


### **Inference in Graphical Models**

Inference turns a factored representation into an answer. The main tasks are:

- **marginal inference:** compute $p(X_i\mid e)$ or $p(X_A\mid e)$;
- **evidence evaluation:** compute $p(e)$, often needed for likelihood and model comparison;
- **MAP inference:** find the most probable values of selected variables;
- **MPE inference:** find the most probable complete assignment;
- **expectation:** compute posterior feature counts used by EM or gradient learning.

The graph may contain only small factors while the requested answer still requires a global computation. Inference algorithms differ in how they organize the required sums and maximizations.

#### **Variable Elimination**

**Variable elimination (VE)** generalizes dynamic programming to arbitrary factor graphs. It uses distributivity to move sums inside products and removes variables one at a time. For example,

$$
p(a)
=\sum_b\sum_c f_1(a,b)f_2(b,c)f_3(c)
=\sum_b f_1(a,b)\left[\sum_c f_2(b,c)f_3(c)\right].
$$

The bracketed expression is an intermediate factor over $b$ and can be reused. A VE step is:

```text
VARIABLE-ELIMINATION(factors, query, evidence, order)
    restrict every factor using the observed evidence
    for variable z in order:
        bucket <- all factors whose scope contains z
        product <- multiply every factor in bucket
        message <- sum product over z
        replace bucket by message
    result <- multiply remaining factors
    return normalize(result over query)
```

Elimination order does not change the exact answer, but it can radically change the largest intermediate factor. Eliminating a highly connected variable early may connect all of its neighbors, a process called **fill-in**. Runtime is exponential in the induced width of the chosen order; the minimum possible induced width is related to graph **treewidth**. Finding the optimal order is itself hard, so practical systems use heuristics such as min-degree or min-fill.

<div class="diagram-scroll">

![Variable elimination collects factors, sums out a variable, and passes the resulting factor forward; belief propagation sends analogous local messages.](assets/variable-elimination-messages.svg){fig-alt="Variable elimination and message passing as two organizations of repeated factor multiplication and marginalization."}

</div>

<details>
<summary><strong>Python: exact variable elimination with tabular factors</strong></summary>

```python
from dataclasses import dataclass
import numpy as np

@dataclass
class Factor:
    variables: tuple
    values: np.ndarray

def restrict(factor, evidence):
    variables = list(factor.variables)
    values = factor.values
    # Slice observed axes from right to left so earlier axis positions stay valid.
    for variable, observed_value in evidence.items():
        if variable in variables:
            axis = variables.index(variable)
            values = np.take(values, observed_value, axis=axis)
            variables.pop(axis)
    return Factor(tuple(variables), np.asarray(values))

def multiply(left, right):
    union = tuple(dict.fromkeys(left.variables + right.variables))

    def align(factor):
        present = [v for v in union if v in factor.variables]
        permutation = [factor.variables.index(v) for v in present]
        values = np.transpose(factor.values, permutation) if permutation else factor.values
        shape = [2 if v in factor.variables else 1 for v in union]
        return np.reshape(values, shape)

    return Factor(union, align(left) * align(right))

def sum_out(factor, variable):
    axis = factor.variables.index(variable)
    remaining = factor.variables[:axis] + factor.variables[axis + 1:]
    return Factor(remaining, factor.values.sum(axis=axis))

def variable_elimination(factors, query, evidence, order):
    working = [restrict(factor, evidence) for factor in factors]
    for variable in order:
        bucket = [factor for factor in working if variable in factor.variables]
        working = [factor for factor in working if variable not in factor.variables]
        if not bucket:
            continue
        product_factor = bucket[0]
        for factor in bucket[1:]:
            product_factor = multiply(product_factor, factor)
        working.append(sum_out(product_factor, variable))

    result = working[0]
    for factor in working[1:]:
        result = multiply(result, factor)
    for variable in tuple(v for v in result.variables if v != query):
        result = sum_out(result, variable)
    return result.values / result.values.sum()

# Bayesian network A -> B -> C, represented by p(A), p(B|A), p(C|B).
factors = [
    Factor(("A",), np.array([0.7, 0.3])),
    Factor(("A", "B"), np.array([[0.9, 0.1], [0.2, 0.8]])),
    Factor(("B", "C"), np.array([[0.85, 0.15], [0.1, 0.9]])),
]

posterior = variable_elimination(
    factors, query="A", evidence={"C": 1}, order=["B"]
)
print("P(A | C=1):", np.round(posterior, 4))
```

</details>

The implementation is intentionally small and assumes binary variables, but the workflow is general: restrict evidence, collect a bucket, multiply, marginalize, and continue. Production libraries add sparse representations, arbitrary cardinalities, stable log-space arithmetic, and optimized order selection.

#### **Belief Propagation**

**Belief propagation (BP)** reorganizes the same local sums as messages between neighboring nodes. In a factor graph, a variable-to-factor message multiplies incoming messages from other factors,

$$
m_{x\rightarrow f}(x)
=\prod_{h\in N(x)\setminus f}m_{h\rightarrow x}(x),
$$

while a factor-to-variable message multiplies the factor by incoming messages and sums out all other variables in its scope:

$$
m_{f\rightarrow x}(x)
=\sum_{\mathbf{x}_{N(f)\setminus x}}
f(\mathbf{x}_{N(f)})
\prod_{y\in N(f)\setminus x}m_{y\rightarrow f}(y).
$$

After messages arrive from every direction, a variable belief is proportional to their product. On trees, sum-product BP gives exact marginals after a collect-and-distribute pass. Replacing sums with maxima gives max-product messages for MAP decoding, together with backpointers when the maximizing assignment must be recovered.

<details>
<summary><strong>Python: sum-product messages on a binary chain</strong></summary>

```python
import numpy as np

# Unary evidence potentials phi_t(x_t) and a shared pairwise potential psi.
unary = np.array([
    [0.9, 0.1],
    [0.4, 0.6],
    [0.2, 0.8],
    [0.7, 0.3],
])
pairwise = np.array([[2.0, 0.5], [0.5, 2.0]])  # favors equal neighbors

length = len(unary)
forward = np.ones((length, 2))
backward = np.ones((length, 2))

for t in range(1, length):
    # Sum over x_{t-1}; result is a message indexed by x_t.
    forward[t] = (forward[t - 1] * unary[t - 1]) @ pairwise
    forward[t] /= forward[t].sum()

for t in range(length - 2, -1, -1):
    # Sum over x_{t+1}; result is a message indexed by x_t.
    backward[t] = pairwise @ (unary[t + 1] * backward[t + 1])
    backward[t] /= backward[t].sum()

beliefs = unary * forward * backward
beliefs /= beliefs.sum(axis=1, keepdims=True)
print("P(X_t=0), P(X_t=1) at each position:")
print(np.round(beliefs, 3))
```

</details>

On graphs with cycles, **loopy belief propagation** repeatedly updates messages and is approximate. It often works well, but convergence is not guaranteed and converged beliefs need not be exact. Message damping, update scheduling, and residual monitoring can improve stability.

#### **Sampling and Variational Methods**

When exact factors become too large, two broad approximation families dominate:

- **Monte Carlo methods** represent a distribution with samples. Importance sampling, Gibbs sampling, and other Markov chain Monte Carlo methods are asymptotically exact under suitable conditions, but can have high variance or slow mixing.
- **Variational inference** chooses a tractable family $q(\mathbf z)$ and optimizes it to approximate the posterior, often by maximizing the evidence lower bound. It is usually faster and deterministic, but its answer is biased by the chosen family and divergence.

For a mean-field approximation $q(\mathbf{x})=\prod_i q_i(x_i)$ to an Ising model, coordinate updates take the form

$$
m_i\leftarrow\tanh\!\left(h_i+\sum_{j\in N(i)}J_{ij}m_j\right),
$$

where $m_i=\mathbb E_q[X_i]$. The update replaces uncertain neighbors by their current expected spins.

<details>
<summary><strong>Python: compare mean-field marginals with exact enumeration</strong></summary>

```python
from itertools import product
import numpy as np

h = np.array([0.2, -0.1, 0.0, 0.3])
J = np.array([0.8, 0.8, 0.8])

# Coordinate-ascent mean field.
mean_spin = np.zeros(4)
for _ in range(200):
    previous = mean_spin.copy()
    for i in range(4):
        field = h[i]
        if i > 0:
            field += J[i - 1] * mean_spin[i - 1]
        if i < 3:
            field += J[i] * mean_spin[i + 1]
        mean_spin[i] = np.tanh(field)
    if np.max(np.abs(mean_spin - previous)) < 1e-10:
        break

# Exact answer for this tiny model.
states = np.array(list(product([-1, 1], repeat=4)))
energies = -states @ h - np.sum(J * states[:, :-1] * states[:, 1:], axis=1)
probabilities = np.exp(-energies)
probabilities /= probabilities.sum()
exact_mean_spin = probabilities @ states

print("Mean-field E[X]:", np.round(mean_spin, 3))
print("Exact E[X]     :", np.round(exact_mean_spin, 3))
```

</details>

Approximate inference should be checked against exact enumeration on small instances whenever possible. Agreement on tiny cases does not prove large-case accuracy, but disagreement immediately reveals implementation or approximation problems. This habit becomes especially valuable when inference is embedded inside learning, where a biased posterior can silently create a biased gradient.


### **Hidden Markov Models**

A **hidden Markov model (HMM)** is a generative probabilistic model for a sequence of observations $X_{1:T}$ controlled by an unobserved state sequence $Z_{1:T}$. The hidden state summarizes the information from the past that the model retains. HMMs are useful when observations are noisy manifestations of a smaller evolving state: phonemes behind acoustic frames, weather behind sensor readings, regimes behind financial measurements, or biological states behind DNA symbols.

The model makes two central assumptions:

1. **First-order Markov state dynamics:**

   $$
   p(z_t\mid z_{1:t-1})=p(z_t\mid z_{t-1}).
   $$

2. **Output independence:** after the current hidden state is known, the current observation is independent of other states and observations:

   $$
   p(x_t\mid z_{1:T},x_{1:t-1})=p(x_t\mid z_t).
   $$

These assumptions turn an apparently global sequence distribution into local transitions and emissions.

<div class="diagram-scroll">

![An HMM unrolled as a trellis, with hidden states connected through time and each hidden state emitting one observation.](assets/hmm-trellis.svg){fig-alt="A hidden Markov model unrolled across time with transition, emission, forward, backward, and Viterbi computations."}

</div>

#### **State, Transition, and Emission Models**

For $K$ hidden states, a discrete HMM is commonly parameterized by:

- an initial distribution $\pi_k=p(Z_1=k)$;
- a transition matrix $A_{ij}=p(Z_t=j\mid Z_{t-1}=i)$;
- an emission model $B_j(x)=p(X_t=x\mid Z_t=j)$.

The joint probability factorizes as

$$
p(z_{1:T},x_{1:T})
=\pi_{z_1}B_{z_1}(x_1)
\prod_{t=2}^{T}A_{z_{t-1},z_t}B_{z_t}(x_t).
$$

Each row of $A$ is a categorical distribution. The emission can be categorical for symbols, Gaussian for continuous vectors, a mixture distribution, or another conditional density. The state must be rich enough that the Markov assumption becomes reasonable. If duration matters, one can expand the state, use a higher-order HMM, or adopt a hidden semi-Markov model with explicit duration distributions.

<details>
<summary><strong>Python: generate observations from a discrete HMM</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(21)
state_names = np.array(["calm", "storm"])
symbol_names = np.array(["dry", "damp", "wet"])

initial = np.array([0.8, 0.2])
transition = np.array([
    [0.85, 0.15],
    [0.30, 0.70],
])
emission = np.array([
    [0.70, 0.25, 0.05],
    [0.10, 0.30, 0.60],
])

def sample_hmm(length, initial, transition, emission, rng):
    states = np.empty(length, dtype=int)
    observations = np.empty(length, dtype=int)
    states[0] = rng.choice(len(initial), p=initial)
    observations[0] = rng.choice(emission.shape[1], p=emission[states[0]])

    for t in range(1, length):
        states[t] = rng.choice(len(initial), p=transition[states[t - 1]])
        observations[t] = rng.choice(emission.shape[1], p=emission[states[t]])
    return states, observations

states, observations = sample_hmm(12, initial, transition, emission, rng)
print("hidden:", state_names[states].tolist())
print("observed:", symbol_names[observations].tolist())
```

</details>

The code exposes a useful distinction: states are part of the model's explanation but normally absent from the observed training record. In simulation both are available, which makes synthetic data valuable for testing inference and learning code.

#### **Forward-Backward Algorithm**

The likelihood of an observed sequence requires summing over all $K^T$ possible state paths:

$$
p(x_{1:T})=\sum_{z_{1:T}}p(z_{1:T},x_{1:T}).
$$

The **forward algorithm** reduces this exponential sum to $O(TK^2)$ by caching partial sums. Define

$$
\alpha_t(j)=p(x_{1:t},Z_t=j).
$$

Initialization and recurrence are

$$
\alpha_1(j)=\pi_jB_j(x_1),
$$

$$
\alpha_t(j)
=B_j(x_t)\sum_{i=1}^{K}\alpha_{t-1}(i)A_{ij}.
$$

Finally, $p(x_{1:T})=\sum_j\alpha_T(j)$. The inner sum combines every previous state that could transition to $j$; multiplication by the emission incorporates the current observation.

Direct probabilities underflow for long sequences. A scaled implementation normalizes each forward vector by $c_t=\sum_j\tilde\alpha_t(j)$ and accumulates

$$
\log p(x_{1:T})=\sum_{t=1}^{T}\log c_t.
$$

Alternatively, all computations can be performed in log space with log-sum-exp.

<details>
<summary><strong>Python: stable scaled forward likelihood</strong></summary>

```python
import numpy as np

initial = np.array([0.8, 0.2])
transition = np.array([[0.85, 0.15], [0.30, 0.70]])
emission = np.array([[0.70, 0.25, 0.05], [0.10, 0.30, 0.60]])
observations = np.array([0, 1, 2, 2, 1, 0])

def forward_scaled(observations, initial, transition, emission):
    length = len(observations)
    n_states = len(initial)
    alpha = np.zeros((length, n_states))
    scales = np.zeros(length)

    alpha[0] = initial * emission[:, observations[0]]
    scales[0] = alpha[0].sum()
    alpha[0] /= scales[0]

    for t in range(1, length):
        alpha[t] = (alpha[t - 1] @ transition) * emission[:, observations[t]]
        scales[t] = alpha[t].sum()
        alpha[t] /= scales[t]

    log_likelihood = np.log(scales).sum()
    return alpha, scales, log_likelihood

filtered, scales, log_likelihood = forward_scaled(
    observations, initial, transition, emission
)
print("filtered state probabilities:\n", np.round(filtered, 3))
print("log p(x_1:T):", round(log_likelihood, 4))
```

</details>

The normalized forward vector is the **filtering distribution** $p(Z_t\mid x_{1:t})$: the state estimate available online using observations only up to the present.

The backward quantity

$$
\beta_t(i)=p(x_{t+1:T}\mid Z_t=i)
$$

summarizes future evidence. Its recurrence is

$$
\beta_t(i)=\sum_{j=1}^{K}A_{ij}B_j(x_{t+1})\beta_{t+1}(j),
$$

with $\beta_T(i)=1$. Combining forward and backward information gives the **smoothed** marginal

$$
\gamma_t(i)
=p(Z_t=i\mid x_{1:T})
=\frac{\alpha_t(i)\beta_t(i)}{p(x_{1:T})}.
$$

Smoothing can revise an earlier state estimate using later observations. This differs from filtering and from forecasting, which asks about $Z_{t+h}$ or $X_{t+h}$ beyond the observed prefix.

<details>
<summary><strong>Python: forward-backward smoothing with consistent scaling</strong></summary>

```python
import numpy as np

initial = np.array([0.8, 0.2])
transition = np.array([[0.85, 0.15], [0.30, 0.70]])
emission = np.array([[0.70, 0.25, 0.05], [0.10, 0.30, 0.60]])
observations = np.array([0, 1, 2, 2, 1, 0])

def forward_backward(observations, initial, transition, emission):
    length = len(observations)
    n_states = len(initial)
    alpha = np.zeros((length, n_states))
    scales = np.zeros(length)

    alpha[0] = initial * emission[:, observations[0]]
    scales[0] = alpha[0].sum()
    alpha[0] /= scales[0]
    for t in range(1, length):
        alpha[t] = (alpha[t - 1] @ transition) * emission[:, observations[t]]
        scales[t] = alpha[t].sum()
        alpha[t] /= scales[t]

    beta = np.ones((length, n_states))
    for t in range(length - 2, -1, -1):
        beta[t] = transition @ (emission[:, observations[t + 1]] * beta[t + 1])
        beta[t] /= scales[t + 1]  # match the normalization used by alpha

    gamma = alpha * beta
    gamma /= gamma.sum(axis=1, keepdims=True)
    return alpha, gamma

filtered, smoothed = forward_backward(observations, initial, transition, emission)
print("P(storm) from filtering:", np.round(filtered[:, 1], 3))
print("P(storm) after smoothing:", np.round(smoothed[:, 1], 3))
```

</details>

#### **Viterbi Decoding**

Forward-backward computes sums over paths. **Viterbi decoding** finds one path with maximum joint probability:

$$
\hat z_{1:T}
=\arg\max_{z_{1:T}}p(z_{1:T},x_{1:T}).
$$

Define the best partial-path score ending in state $j$:

$$
\delta_t(j)
=\max_{z_{1:t-1}}p(z_{1:t-1},Z_t=j,x_{1:t}).
$$

Then

$$
\delta_t(j)
=B_j(x_t)\max_i\left[\delta_{t-1}(i)A_{ij}\right].
$$

The maximizing predecessor for each $(t,j)$ is stored in a backpointer. After selecting the best final state, the algorithm follows those pointers backward to reconstruct the path. In practice, log probabilities turn multiplication into addition and prevent underflow.

<div class="diagram-scroll">

![Forward and Viterbi use the same trellis but apply sum and maximum aggregation respectively.](assets/forward-viterbi-semiring.svg){fig-alt="Forward and Viterbi dynamic programs compared as sum-product and max-product computations on the same trellis."}

</div>

<details>
<summary><strong>Python: Viterbi decoding with log probabilities and backpointers</strong></summary>

```python
import numpy as np

initial = np.array([0.8, 0.2])
transition = np.array([[0.85, 0.15], [0.30, 0.70]])
emission = np.array([[0.70, 0.25, 0.05], [0.10, 0.30, 0.60]])
observations = np.array([0, 1, 2, 2, 1, 0])

def viterbi(observations, initial, transition, emission):
    length = len(observations)
    n_states = len(initial)
    log_initial = np.log(initial)
    log_transition = np.log(transition)
    log_emission = np.log(emission)

    score = np.full((length, n_states), -np.inf)
    backpointer = np.zeros((length, n_states), dtype=int)
    score[0] = log_initial + log_emission[:, observations[0]]

    for t in range(1, length):
        for current in range(n_states):
            candidates = score[t - 1] + log_transition[:, current]
            backpointer[t, current] = np.argmax(candidates)
            score[t, current] = candidates[backpointer[t, current]] + log_emission[
                current, observations[t]
            ]

    path = np.zeros(length, dtype=int)
    path[-1] = np.argmax(score[-1])
    for t in range(length - 2, -1, -1):
        path[t] = backpointer[t + 1, path[t + 1]]
    return path, score[-1, path[-1]]

path, path_log_probability = viterbi(observations, initial, transition, emission)
print("Viterbi state path:", path.tolist())
print("joint log probability:", round(path_log_probability, 4))
```

</details>

Viterbi decoding is not the same as choosing the most probable state independently at every position. **Posterior decoding** uses $\arg\max_i p(Z_t=i\mid x_{1:T})$ and minimizes expected per-position error, but its selected labels may form an implausible or even invalid transition sequence. Viterbi returns a coherent path and optimizes whole-path probability. The correct decoder depends on the task loss.

#### **Baum-Welch Learning**

When state labels are hidden, direct transition and emission counts are unavailable. **Baum-Welch** is the EM algorithm specialized to HMMs:

1. **E-step:** run forward-backward under the current parameters to compute expected state occupancy

   $$
   \gamma_t(i)=p(Z_t=i\mid x_{1:T})
   $$

   and expected transition use

   $$
   \xi_t(i,j)=p(Z_t=i,Z_{t+1}=j\mid x_{1:T}).
   $$

2. **M-step:** normalize expected counts:

   $$
   \pi_i^{\mathrm{new}}=\gamma_1(i),
   $$

   $$
   A_{ij}^{\mathrm{new}}
   =\frac{\sum_{t=1}^{T-1}\xi_t(i,j)}
   {\sum_{t=1}^{T-1}\gamma_t(i)},
   $$

   $$
   B_i^{\mathrm{new}}(v)
   =\frac{\sum_{t=1}^{T}\gamma_t(i)\mathbb 1[x_t=v]}
   {\sum_{t=1}^{T}\gamma_t(i)}.
   $$

Each iteration does not decrease the observed-data likelihood, but EM can converge to a local optimum. Hidden-state labels are also permutation-invariant: swapping state names leaves the observed distribution unchanged. Multiple initializations and meaningful constraints are therefore important.

<details>
<summary><strong>Python: one-sequence Baum-Welch for a categorical HMM</strong></summary>

```python
import numpy as np

observations = np.array([0, 0, 1, 2, 2, 2, 1, 0, 0, 1, 2, 2])
n_states = 2
n_symbols = 3
rng = np.random.default_rng(4)

def normalize_rows(matrix):
    return matrix / matrix.sum(axis=1, keepdims=True)

initial = np.array([0.5, 0.5])
transition = normalize_rows(rng.random((n_states, n_states)))
emission = normalize_rows(rng.random((n_states, n_symbols)))

def expectation_step(obs, initial, transition, emission):
    length = len(obs)
    alpha = np.zeros((length, n_states))
    scales = np.zeros(length)
    alpha[0] = initial * emission[:, obs[0]]
    scales[0] = alpha[0].sum()
    alpha[0] /= scales[0]

    for t in range(1, length):
        alpha[t] = (alpha[t - 1] @ transition) * emission[:, obs[t]]
        scales[t] = alpha[t].sum()
        alpha[t] /= scales[t]

    beta = np.ones((length, n_states))
    for t in range(length - 2, -1, -1):
        beta[t] = transition @ (emission[:, obs[t + 1]] * beta[t + 1])
        beta[t] /= scales[t + 1]

    gamma = alpha * beta
    gamma /= gamma.sum(axis=1, keepdims=True)

    xi = np.zeros((length - 1, n_states, n_states))
    for t in range(length - 1):
        xi[t] = (
            alpha[t, :, None]
            * transition
            * emission[:, obs[t + 1]][None, :]
            * beta[t + 1][None, :]
        )
        xi[t] /= xi[t].sum()
    return gamma, xi, np.log(scales).sum()

likelihood_history = []
for _ in range(30):
    gamma, xi, log_likelihood = expectation_step(
        observations, initial, transition, emission
    )
    likelihood_history.append(log_likelihood)

    initial = gamma[0]
    transition = xi.sum(axis=0) / gamma[:-1].sum(axis=0)[:, None]
    for symbol in range(n_symbols):
        emission[:, symbol] = gamma[observations == symbol].sum(axis=0)
    emission /= gamma.sum(axis=0)[:, None]

print("first/last log likelihood:", round(likelihood_history[0], 3),
      round(likelihood_history[-1], 3))
print("learned transition:\n", np.round(transition, 3))
print("learned emission:\n", np.round(emission, 3))
```

</details>

For multiple independent sequences, expected counts are summed across sequences before the M-step. Padding must not be treated as observations; masks should exclude padded positions and transitions across sequence boundaries. Supervised HMM training is simpler because known state sequences provide ordinary counts.

| Question | Algorithm | Aggregation | Output |
|---|---|---|---|
| How likely is the observation sequence? | Forward | Sum over paths | $p(x_{1:T})$ |
| What state was likely at each time using all evidence? | Forward-backward | Sum over paths | $p(z_t\mid x_{1:T})$ |
| What is the single most probable path? | Viterbi | Maximum over paths | $\arg\max_{z_{1:T}}p(z_{1:T},x_{1:T})$ |
| How can unlabeled parameters be fitted? | Baum-Welch | Expected sufficient statistics | Updated $\pi,A,B$ |


### **Conditional Random Fields**

An HMM explains both observations and labels through a joint model $p(\mathbf{x},\mathbf{y})$. This requires an emission distribution for the input and conditional-independence assumptions such as $X_t\perp X_{s}\mid Y_t$. In many prediction tasks the observation is a rich object with overlapping, correlated features. A **conditional random field (CRF)** avoids modeling how the input was generated and directly models

$$
p(\mathbf{y}\mid\mathbf{x}).
$$

For named-entity recognition, $\mathbf{x}$ can include words, capitalization, affixes, dictionary matches, and contextual embeddings. The CRF needs to model compatibility among output labels, but it does not need a probability distribution over all possible sentences or feature vectors.

<div class="diagram-scroll">

![An HMM generates observations from hidden states, while a linear-chain CRF conditions the entire label sequence on the observed input.](assets/hmm-crf-comparison.svg){fig-alt="A comparison between the directed generative factorization of an HMM and the conditional globally normalized factorization of a linear-chain CRF."}

</div>

#### **Local versus Global Normalization**

A locally normalized sequence model predicts each next label with a conditional distribution such as

$$
p(y_t\mid y_{t-1},\mathbf{x}).
$$

Each state distributes one unit of probability among its own outgoing transitions. A state with only one allowed successor must assign that edge probability one, regardless of later evidence. This can create **label bias**: probability becomes trapped in parts of the state graph with fewer outgoing choices.

A linear-chain CRF assigns an unnormalized score to each complete label sequence and normalizes once across all valid sequences:

$$
p(\mathbf y\mid\mathbf x)
=\frac{\exp s_\mathbf{w}(\mathbf{x},\mathbf{y})}
{Z_\mathbf{w}(\mathbf{x})},
$$

$$
Z_\mathbf{w}(\mathbf{x})
=\sum_{\mathbf{y}'\in\mathcal Y(\mathbf{x})}
\exp s_\mathbf{w}(\mathbf{x},\mathbf{y}').
$$

Because every sequence competes in the same denominator, evidence at a later position can influence preferences at earlier positions. Global normalization does not guarantee a better model in every setting, but it removes the specific asymmetry caused by per-state normalization.

For feature functions $f_k(y_{t-1},y_t,\mathbf{x},t)$, a common score is

$$
s_\mathbf{w}(\mathbf{x},\mathbf{y})
=\sum_{t=1}^{T}\sum_{k=1}^{m}
w_k f_k(y_{t-1},y_t,\mathbf{x},t).
$$

A positive weight means the corresponding feature increases sequence compatibility. Features may depend on the entire observed input because $\mathbf{x}$ is conditioned on, while tractable decoding requires the label dependence to remain local.

#### **Linear-Chain CRFs**

A convenient neural or feature-based parameterization separates a unary emission score $e_t(j;\mathbf{x})$ from a transition score $A_{ij}$:

$$
s(\mathbf{x},\mathbf{y})
=\sum_{t=1}^{T}e_t(y_t;\mathbf{x})
+\sum_{t=2}^{T}A_{y_{t-1},y_t}.
$$

The unary scores may come from hand-crafted features, a linear classifier, an LSTM, or a Transformer. The CRF layer adds a globally coherent output distribution and trainable transition preferences.

The log-partition function is computed with a forward recurrence in log space:

$$
\alpha_1(j)=e_1(j),
$$

$$
\alpha_t(j)
=e_t(j)+\operatorname{LSE}_i\left[\alpha_{t-1}(i)+A_{ij}\right],
$$

$$
\log Z(\mathbf{x})=\operatorname{LSE}_j\alpha_T(j),
$$

where

$$
\operatorname{LSE}(a_1,\ldots,a_K)
=m+\log\sum_k e^{a_k-m},
\qquad m=\max_k a_k,
$$

is the numerically stable log-sum-exp operation.

<details>
<summary><strong>Python: compute CRF sequence probability and log-partition</strong></summary>

```python
import numpy as np

def logsumexp(values, axis=None):
    maximum = np.max(values, axis=axis, keepdims=True)
    result = maximum + np.log(np.sum(np.exp(values - maximum), axis=axis, keepdims=True))
    return np.squeeze(result, axis=axis) if axis is not None else result.item()

def crf_log_partition(unary_scores, transition_scores):
    alpha = unary_scores[0].copy()
    for t in range(1, len(unary_scores)):
        candidates = alpha[:, None] + transition_scores
        alpha = unary_scores[t] + logsumexp(candidates, axis=0)
    return logsumexp(alpha)

def sequence_score(labels, unary_scores, transition_scores):
    score = unary_scores[np.arange(len(labels)), labels].sum()
    score += transition_scores[labels[:-1], labels[1:]].sum()
    return score

# Three positions and two labels. Scores need not already be probabilities.
unary = np.array([[1.5, -0.2], [0.1, 1.0], [0.4, 1.2]])
transition = np.array([[0.7, -0.4], [-0.2, 0.6]])
labels = np.array([0, 1, 1])

log_z = crf_log_partition(unary, transition)
log_probability = sequence_score(labels, unary, transition) - log_z
print("log Z(x):", round(log_z, 4))
print("p(y|x):", round(np.exp(log_probability), 4))
```

</details>

For a labeled pair $(\mathbf{x},\mathbf{y})$, the negative conditional log-likelihood is

$$
\mathcal L(\mathbf w)
=-s_\mathbf w(\mathbf{x},\mathbf{y})
+\log Z_\mathbf w(\mathbf{x}).
$$

Its gradient has an interpretable form:

$$
\nabla_\mathbf w\log p(\mathbf y\mid\mathbf x)
=\Phi(\mathbf{x},\mathbf{y})
-\mathbb E_{p_\mathbf w(\mathbf y'\mid\mathbf x)}
[\Phi(\mathbf{x},\mathbf{y}')].
$$

Training increases weights for features observed in the gold sequence and decreases them according to how often the current model expects those features. Forward-backward computes the node and edge marginals needed for expected feature counts.

<details>
<summary><strong>Python: CRF node and transition marginals</strong></summary>

```python
import numpy as np

def logsumexp(values, axis=None):
    maximum = np.max(values, axis=axis, keepdims=True)
    result = maximum + np.log(np.sum(np.exp(values - maximum), axis=axis, keepdims=True))
    return np.squeeze(result, axis=axis) if axis is not None else result.item()

unary = np.array([[1.5, -0.2], [0.1, 1.0], [0.4, 1.2]])
transition = np.array([[0.7, -0.4], [-0.2, 0.6]])
length, n_labels = unary.shape

forward = np.zeros((length, n_labels))
forward[0] = unary[0]
for t in range(1, length):
    forward[t] = unary[t] + logsumexp(
        forward[t - 1, :, None] + transition, axis=0
    )

backward = np.zeros((length, n_labels))
for t in range(length - 2, -1, -1):
    backward[t] = logsumexp(
        transition + unary[t + 1][None, :] + backward[t + 1][None, :], axis=1
    )

log_z = logsumexp(forward[-1])
node_marginals = np.exp(forward + backward - log_z)

edge_marginals = []
for t in range(length - 1):
    log_edge = (
        forward[t, :, None]
        + transition
        + unary[t + 1][None, :]
        + backward[t + 1][None, :]
        - log_z
    )
    edge_marginals.append(np.exp(log_edge))

print("node marginals:\n", np.round(node_marginals, 3))
print("expected transition counts:\n", np.round(np.sum(edge_marginals, axis=0), 3))
```

</details>

For MAP prediction, replacing log-sum-exp with maximum yields Viterbi decoding over CRF scores. This is the same dynamic-programming structure as HMM Viterbi, but the scores have a different meaning: CRF scores are conditional feature compatibilities rather than log initial, transition, and emission probabilities from a joint generative model.

<details>
<summary><strong>Python: decode a linear-chain CRF</strong></summary>

```python
import numpy as np

unary = np.array([[1.5, -0.2], [0.1, 1.0], [0.4, 1.2]])
transition = np.array([[0.7, -0.4], [-0.2, 0.6]])
length, n_labels = unary.shape

best_score = np.zeros((length, n_labels))
backpointer = np.zeros((length, n_labels), dtype=int)
best_score[0] = unary[0]

for t in range(1, length):
    candidates = best_score[t - 1, :, None] + transition
    backpointer[t] = np.argmax(candidates, axis=0)
    best_score[t] = unary[t] + np.max(candidates, axis=0)

best_path = np.zeros(length, dtype=int)
best_path[-1] = np.argmax(best_score[-1])
for t in range(length - 2, -1, -1):
    best_path[t] = backpointer[t + 1, best_path[t + 1]]

print("best labels:", best_path.tolist())
print("best score:", round(best_score[-1, best_path[-1]], 3))
```

</details>

The chain structure gives $O(TK^2)$ training and decoding. If arbitrary long-range label interactions are added, exact inference may become intractable. Practical CRFs balance feature richness on the observed input against carefully restricted structure on the output.

| Property | HMM | Linear-chain CRF |
|---|---|---|
| Objective | Joint $p(\mathbf{x},\mathbf{y})$ | Conditional $p(\mathbf{y}\mid\mathbf{x})$ |
| Input assumptions | Must define emissions and their conditional independence | Can use overlapping input features freely |
| Normalization | Local stochastic matrices | One partition function per input sequence |
| Latent-state use | Natural | Possible but more complex |
| Data efficiency | Can exploit generative assumptions and unlabeled observations | Usually needs labeled input-output sequences |
| Shared algorithmic core | Forward-backward and Viterbi | Forward-backward and Viterbi over feature scores |


### **Structured Prediction**

**Structured prediction** is supervised learning in which the output $\mathbf y$ has dependent parts: a sequence, tree, matching, segmentation, ranking, or another combinatorial object. The output space can be enormous. A sequence of length $T$ with $K$ labels has $K^T$ possible assignments, so treating every sequence as an unrelated class is impossible.

Most structured predictors define a compatibility score

$$
s_\mathbf w(\mathbf{x},\mathbf{y})
=\mathbf w^\top\Phi(\mathbf{x},\mathbf{y}),
$$

and predict

$$
\widehat{\mathbf y}
=\arg\max_{\mathbf y\in\mathcal Y(\mathbf{x})}
s_\mathbf w(\mathbf{x},\mathbf y).
$$

$\Phi(\mathbf{x},\mathbf y)$ is a global feature vector that usually decomposes over local parts. The **decoder** exploits that decomposition with Viterbi, CKY parsing, a shortest-path method, a matching algorithm, integer programming, or approximate search. The same learning principle can therefore be used with different output structures as long as the required inference oracle is available.

<div class="diagram-scroll">

![Structured learning alternates between scoring candidate outputs, performing structured inference, comparing the prediction with the gold structure, and updating parameters.](assets/structured-prediction-training.svg){fig-alt="A structured prediction training loop built around a reusable inference oracle."}

</div>

#### **Structured Perceptron and Structured SVM**

The **structured perceptron** extends the binary perceptron. For training pair $(\mathbf{x}_i,\mathbf{y}_i)$, it first predicts the highest-scoring structure

$$
\widehat{\mathbf y}_i
=\arg\max_{\mathbf y}s_\mathbf w(\mathbf{x}_i,\mathbf y),
$$

then updates

$$
\mathbf w
\leftarrow
\mathbf w
+\eta\left[
\Phi(\mathbf{x}_i,\mathbf y_i)
-\Phi(\mathbf{x}_i,\widehat{\mathbf y}_i)
\right].
$$

The update rewards features of the gold structure and penalizes features of the mistaken prediction. It does not compute a partition function and needs only MAP decoding. Averaging parameters across updates usually improves generalization and reduces oscillation.

<details>
<summary><strong>Python: train a structured perceptron for sequence labeling</strong></summary>

```python
import numpy as np

# Each sequence contains scalar observations; labels are 0 or 1.
training_data = [
    (np.array([-1.2, -0.7, 0.8, 1.1]), np.array([0, 0, 1, 1])),
    (np.array([-0.9, 0.4, 0.7]), np.array([0, 1, 1])),
    (np.array([1.0, 0.6, -0.5]), np.array([1, 1, 0])),
]

n_labels = 2
unary_weights = np.zeros(n_labels)       # score label j by w_j * x_t
transition_weights = np.zeros((n_labels, n_labels))

def decode(observations, unary_weights, transition_weights):
    unary_scores = observations[:, None] * unary_weights[None, :]
    length = len(observations)
    score = np.zeros((length, n_labels))
    backpointer = np.zeros((length, n_labels), dtype=int)
    score[0] = unary_scores[0]

    for t in range(1, length):
        candidates = score[t - 1, :, None] + transition_weights
        backpointer[t] = np.argmax(candidates, axis=0)
        score[t] = unary_scores[t] + np.max(candidates, axis=0)

    path = np.zeros(length, dtype=int)
    path[-1] = np.argmax(score[-1])
    for t in range(length - 2, -1, -1):
        path[t] = backpointer[t + 1, path[t + 1]]
    return path

def feature_counts(observations, labels):
    unary = np.zeros(n_labels)
    transitions = np.zeros((n_labels, n_labels))
    for x_t, y_t in zip(observations, labels):
        unary[y_t] += x_t
    for previous, current in zip(labels[:-1], labels[1:]):
        transitions[previous, current] += 1
    return unary, transitions

average_unary = np.zeros_like(unary_weights)
average_transition = np.zeros_like(transition_weights)
n_updates = 0

for epoch in range(15):
    for observations, gold in training_data:
        prediction = decode(observations, unary_weights, transition_weights)
        if not np.array_equal(prediction, gold):
            gold_unary, gold_transition = feature_counts(observations, gold)
            pred_unary, pred_transition = feature_counts(observations, prediction)
            unary_weights += gold_unary - pred_unary
            transition_weights += gold_transition - pred_transition

        average_unary += unary_weights
        average_transition += transition_weights
        n_updates += 1

average_unary /= n_updates
average_transition /= n_updates
test = np.array([-0.8, -0.2, 0.9, 0.6])
print("predicted labels:", decode(test, average_unary, average_transition).tolist())
```

</details>

The example is deliberately transparent: unary features connect each observation to a label, transition features reward label pairs, and Viterbi serves as the inference oracle. Real systems use larger sparse feature vectors or neural unary scores, but the update logic is unchanged.

A **structured support vector machine (structured SVM)** adds a margin that depends on how wrong a candidate structure is. One common per-example hinge loss is

$$
\mathcal L_i(\mathbf w)
=\max_{\mathbf y\in\mathcal Y(\mathbf{x}_i)}
\left[
\Delta(\mathbf y_i,\mathbf y)
+s_\mathbf w(\mathbf{x}_i,\mathbf y)
-s_\mathbf w(\mathbf{x}_i,\mathbf y_i)
\right],
$$

where $\Delta$ may be Hamming loss, span loss, or another task cost. The model must separate the gold output from a bad output by a larger margin than from a nearly correct one. Unlike the perceptron, this objective explicitly encodes the severity of each structured error and usually includes regularization.

#### **Loss-Augmented Inference**

Training a structured SVM requires finding the most violated output:

$$
\widetilde{\mathbf y}
=\arg\max_{\mathbf y}
\left[s_\mathbf w(\mathbf{x},\mathbf y)
+\Delta(\mathbf y_{\mathrm{gold}},\mathbf y)\right].
$$

This is **loss-augmented inference**. It deliberately rewards candidates for being wrong so that training concentrates on errors that are both high-scoring and costly. If $\Delta$ decomposes like the model score, the ordinary decoder can often be reused after modifying local scores.

For token-level Hamming loss,

$$
\Delta(\mathbf y_{\mathrm{gold}},\mathbf y)
=\sum_t\mathbb 1[y_t\neq y_t^{\mathrm{gold}}],
$$

one simply adds $1$ to the unary score of every incorrect label at each position before Viterbi decoding.

<details>
<summary><strong>Python: loss-augmented Viterbi and a structured hinge</strong></summary>

```python
import numpy as np

unary = np.array([
    [1.2, 0.8],
    [0.4, 1.1],
    [0.9, 0.6],
])
transition = np.array([[0.5, -0.1], [-0.2, 0.4]])
gold = np.array([0, 1, 0])

def viterbi_with_score(unary_scores, transition_scores):
    length, n_labels = unary_scores.shape
    score = np.zeros((length, n_labels))
    backpointer = np.zeros((length, n_labels), dtype=int)
    score[0] = unary_scores[0]
    for t in range(1, length):
        candidates = score[t - 1, :, None] + transition_scores
        backpointer[t] = np.argmax(candidates, axis=0)
        score[t] = unary_scores[t] + np.max(candidates, axis=0)
    path = np.zeros(length, dtype=int)
    path[-1] = np.argmax(score[-1])
    for t in range(length - 2, -1, -1):
        path[t] = backpointer[t + 1, path[t + 1]]
    return path, score[-1, path[-1]]

def path_score(path, unary_scores, transition_scores):
    value = unary_scores[np.arange(len(path)), path].sum()
    value += transition_scores[path[:-1], path[1:]].sum()
    return value

# Add decomposable Hamming loss to every non-gold unary label.
loss_augmented_unary = unary.copy()
for t, gold_label in enumerate(gold):
    loss_augmented_unary[t, np.arange(unary.shape[1]) != gold_label] += 1.0

violator, augmented_score = viterbi_with_score(loss_augmented_unary, transition)
hamming_loss = np.sum(violator != gold)
hinge = max(0.0, hamming_loss + path_score(violator, unary, transition)
            - path_score(gold, unary, transition))

print("most violated sequence:", violator.tolist())
print("Hamming loss:", int(hamming_loss))
print("structured hinge:", round(hinge, 3))
```

</details>

The structured SVM provides no calibrated probability. It is attractive when an efficient decoder exists and the task loss matters more than probabilistic uncertainty. A CRF is preferable when conditional likelihood, marginals, or uncertainty estimates are needed. Both rely on the same structural decomposition, but optimize different training criteria.

| Method | Training signal | Inference needed during training | Probability output | Main strength |
|---|---|---|---:|---|
| Linear-chain CRF | Conditional log-likelihood | Sum-product and MAP | Yes | Globally normalized uncertainty |
| Structured perceptron | Gold minus predicted features | MAP | No | Simple and fast online updates |
| Structured SVM | Cost-sensitive margin | Loss-augmented MAP | No | Direct control of structured error severity |


### **Sequential Data and Time-Aware Evaluation**

Structure changes evaluation as well as modeling. Randomly shuffling rows is valid only when rows are exchangeable units and no information crosses the split. Sequential projects violate this assumption in several ways:

- adjacent windows from one long recording overlap and contain nearly identical observations;
- multiple sequences belong to the same person, device, document, or session;
- features use future context that would not exist at deployment time;
- the data-generating process changes, so future examples differ from the past;
- normalization, vocabulary construction, imputation, or representation learning is fitted before splitting.

The split unit should match the independent deployment unit. If the model will generalize to unseen patients, split by patient. If it predicts later events for known devices, preserve chronological order within device. If it labels complete documents offline and documents are independent, document-level random splitting may be acceptable even though tokens inside each document are sequential.

<div class="diagram-scroll">

![Random, grouped, chronological, and rolling-origin evaluation schemes for sequential data.](assets/sequential-evaluation.svg){fig-alt="Evaluation protocols that distinguish random splitting from group-aware and time-respecting splits."}

</div>

For forecasting or online prediction, training must precede validation in time. Two common protocols are:

- **expanding window:** train on $[1,t]$, validate on $(t,t+h]$, then enlarge the training history;
- **rolling window:** train on the most recent fixed-width interval, validate on the next horizon, then move both windows forward.

The expanding window uses all history but may be slow to adapt under drift. The rolling window discards stale observations but has less training data. The horizon $h$, update frequency, and any gap between train and validation should reproduce deployment latency.

<details>
<summary><strong>Python: construct expanding-window splits without future leakage</strong></summary>

```python
import numpy as np

def expanding_window_splits(n_samples, initial_train, horizon, step):
    """Yield index arrays whose validation interval is strictly after training."""
    train_end = initial_train
    while train_end + horizon <= n_samples:
        train_index = np.arange(0, train_end)
        validation_index = np.arange(train_end, train_end + horizon)
        yield train_index, validation_index
        train_end += step

timestamps = np.arange(30)
for fold, (train_idx, validation_idx) in enumerate(
    expanding_window_splits(len(timestamps), initial_train=12, horizon=4, step=4),
    start=1,
):
    print(
        f"fold {fold}: train {train_idx[0]}..{train_idx[-1]}, "
        f"validate {validation_idx[0]}..{validation_idx[-1]}"
    )
```

</details>

Preprocessing must be refitted inside every training fold. For example, a tokenizer vocabulary learned from all documents exposes future word types; standardization over the full timeline leaks future means and variances; target-derived features can leak labels directly. A pipeline should learn every data-dependent transformation from the training portion and apply it unchanged to validation.

Metrics must also respect output structure. For sequence labeling, at least three levels are useful:

1. **token accuracy or token F1** asks whether each position is labeled correctly;
2. **span-level precision, recall, and F1** requires entity boundaries and types to match under a declared convention;
3. **exact-sequence accuracy** requires every position in the sequence to be correct.

These answer different questions. A model can have high token accuracy because the background label dominates while missing rare entities. It can have good token F1 but poor exact sequence accuracy on long sequences. For BIO-style labeling, an invalid transition such as `O -> I-PER` also needs an explicit repair or scoring policy.

For generative sequence models, average negative log-likelihood should be normalized consistently by token, frame, or sequence. Perplexity is comparable only when tokenization and evaluation units match. For state recovery in a latent model, raw state IDs cannot be compared directly because latent labels can be permuted; align states or evaluate observable likelihood and downstream utility.

<details>
<summary><strong>Python: compare token, span, and exact-sequence metrics</strong></summary>

```python
from collections import Counter

gold_sequences = [
    ["B-PER", "I-PER", "O", "B-LOC"],
    ["O", "B-ORG", "I-ORG", "O"],
]
predicted_sequences = [
    ["B-PER", "I-PER", "O", "O"],
    ["O", "B-ORG", "O", "O"],
]

def bio_spans(labels):
    """Return (type, start, end-exclusive) spans; invalid I-tags start a new span."""
    spans = []
    start = None
    entity_type = None
    for index, label in enumerate(labels + ["O"]):
        prefix, _, current_type = label.partition("-")
        continues = prefix == "I" and start is not None and current_type == entity_type
        if start is not None and not continues:
            spans.append((entity_type, start, index))
            start = None
            entity_type = None
        if prefix == "B" or (prefix == "I" and not continues):
            start = index
            entity_type = current_type
    return set(spans)

gold_tokens = [label for sequence in gold_sequences for label in sequence]
predicted_tokens = [label for sequence in predicted_sequences for label in sequence]
token_accuracy = sum(g == p for g, p in zip(gold_tokens, predicted_tokens)) / len(gold_tokens)
exact_sequence_accuracy = sum(
    gold == predicted for gold, predicted in zip(gold_sequences, predicted_sequences)
) / len(gold_sequences)

gold_spans = {(i, *span) for i, seq in enumerate(gold_sequences) for span in bio_spans(seq)}
predicted_spans = {
    (i, *span) for i, seq in enumerate(predicted_sequences) for span in bio_spans(seq)
}
true_positive = len(gold_spans & predicted_spans)
precision = true_positive / len(predicted_spans)
recall = true_positive / len(gold_spans)
span_f1 = 2 * precision * recall / (precision + recall)

print("token accuracy:", round(token_accuracy, 3))
print("span F1:", round(span_f1, 3))
print("exact-sequence accuracy:", round(exact_sequence_accuracy, 3))
```

</details>

Evaluation should report variability across appropriate units. If sequences come from users, confidence intervals should resample users rather than individual tokens. When class frequencies, sequence lengths, domains, or time periods differ, report stratified metrics rather than one aggregate that hides the failure mode.

A defensible sequence evaluation checklist is:

1. define what information is available at prediction time;
2. choose the sequence, group, and time unit that must remain isolated;
3. fit preprocessing only on each training split;
4. select metrics at the same structural level as the real error cost;
5. compare against a non-structured baseline to measure the value of dependence modeling;
6. inspect performance by length, label frequency, group, and time period;
7. preserve one untouched final test interval or collection.


### **Choosing a Probabilistic Structured Model**

The first model-selection question is not “Which graphical model is strongest?” but “Which structured uncertainty must the system represent?” A graph should encode the smallest dependency pattern that supports the task. Extra structure adds parameters and inference cost; missing structure creates systematic errors.

Use the following sequence of decisions:

1. **Is the output structured?** If labels are conditionally independent given a strong encoder and independent predictions satisfy the task constraints, a standard classifier may be enough. Always establish this baseline.
2. **Must the input distribution or latent process be modeled?** Use an HMM or another generative latent-variable model when sequence likelihood, missing observations, simulation, unsupervised state discovery, or partial labeling is central.
3. **Is the input richly observed and the target fully labeled?** A CRF is often preferable when overlapping input features and coherent adjacent labels matter.
4. **Are calibrated marginals required?** Choose a normalized probabilistic model and evaluate calibration. Structured perceptrons and SVMs return scores, not probabilities.
5. **Does the task loss decompose?** A structured SVM is attractive when loss-augmented inference remains tractable and a task-specific margin matters.
6. **What is the graph topology?** Chains and trees permit exact dynamic programming. Dense loops or high treewidth may require sampling, variational inference, approximate decoding, or a simpler graph.
7. **What happens at deployment?** Online filtering, offline smoothing, and whole-sequence decoding expose different information and therefore need different inference and evaluation protocols.

| Requirement | Sensible starting point | Why | Main caution |
|---|---|---|---|
| Unsupervised regimes in a sequence | HMM | Explicit latent dynamics and EM learning | State assumptions and local optima |
| Labeled sequence with rich input features | Linear-chain CRF | Conditional global normalization | $O(TK^2)$ and labeled-data requirement |
| Fast structured online updates | Averaged structured perceptron | MAP oracle only | No calibrated probability or explicit margin |
| Task-specific structured cost | Structured SVM | Loss enters the margin directly | Loss-augmented inference must be tractable |
| Symmetric spatial compatibility | Pairwise MRF | Natural undirected interactions | Partition function and loopy inference |
| Diagnostic or causal factorization | Bayesian network | Explicit directed assumptions | Causal meaning needs more than arrows |
| Long-range context with local output constraints | Neural encoder plus CRF | Flexible representation plus exact chain decoding | Transition layer may add little if encoder dominates |

Modern neural networks do not make graphical models obsolete. A Transformer can produce the unary score $e_t(j;\mathbf x)$, while a CRF enforces label transitions; a neural emission model can be embedded in an HMM; and factorized probabilistic structure can express constraints that a generic decoder would otherwise need to rediscover. The useful distinction is between **representation learning**, which constructs informative local scores, and **structured inference**, which combines those scores under global dependencies.

A careful workflow is:

1. specify variables, observed evidence, latent quantities, and the required prediction;
2. draw the proposed graph and state each conditional-independence assumption in words;
3. write the factorization before writing code;
4. identify whether inference requires sums, maxima, expectations, or samples;
5. test the implementation against exhaustive enumeration on a tiny case;
6. separate model score, inference algorithm, training objective, and evaluation loss;
7. compare with a simpler independent baseline and report the incremental value of structure.

The central lesson is that graphical models are not a catalog of isolated algorithms. They are a common language for turning global dependence into local computation. Bayesian networks and MRFs specify the language; variable elimination and message passing execute it; HMMs and CRFs specialize it to sequences; and structured discriminative methods show that the same decoder can support objectives beyond probability. Once representation, inference, learning, and decision are kept separate, it becomes much easier to choose a model whose assumptions and computational cost match the actual problem.

The general graphical-model treatment follows the scope used in [Stanford CS228](https://cs.stanford.edu/~ermon/cs228/index.html), while the sequence algorithms align with the HMM and CRF formulations taught in [MIT OpenCourseWare](https://ocw.mit.edu/courses/6-864-advanced-natural-language-processing-fall-2005/resources/lec6/) and [CMU's graphical-model sequence notes](https://www.cs.cmu.edu/~epxing/Class/10708-16/slide/lecture9-DiscreteSequential.pdf). The diagrams in this chapter are local teaching illustrations so that they remain stable and readable in the rendered blog.
